# Parameter Sensitivity Experiments

Often we want to understand how parameter settings influence our model output. One way to do this is a parameter sensitivity analysis, which quantifies how much the model output changes as parameter settings are adjusted.

There are many types of sensitivity experiments, and they differ in objective and cost. Here we will use an one-at-a-time (OAT) design, perturbing each parameter only to its minimum and maximum values. This approach only requires 2N + 1 simulations for N parameters, and it is easy to interpret since each change in output can be attributed to a single parameter. However, it has limitations, results depend on the default values of the other parameters, it cannot detect parameter interactions, and sampling only the endpoints misses any nonlinear responses. OAT is therefore best viewed as a first screening step to identify which parameters merit further investigation.

In this section we will introduce a new python tool, `meristem`, which assists in creating and managing parameter ensembles for CLM and CLM-FATES. We will use this tool to create and submit ensembles at a site (single-point case), but the tool can work for any configuration of CLM.

---

For more information on one-at-a-time sensitivity experiments with CLM and CLM-FATES see: 

Kennedy, D., Dagon, K., Lawrence, D. M., Fisher, R. A., Sanderson, B. M., Collier, N., Hoffman, F. M., Koven, C. D., Kluzek, E., Levis, S., Lu, X., Oleson, K. W., Zarakas, C. M., Cheng, Y., Foster, A. C., Fowler, M. D., Hawkins, L. R., Kavoo, T., Kumar, S., … Wood, A. W. (2025). One-at-a-Time Parameter Perturbation Ensemble of the Community Land Model, Version 5.1. Journal of Advances in Modeling Earth Systems, 17(8), e2024MS004715. https://doi.org/10.1029/2024MS004715

Foster, A. C., Hawkins, L. R., Kennedy, D., Bonan, G. B., Fisher, R. A., Needham, J. F., Knox, R. G., Koven, C. D., Wieder, W. R., Dagon, K., & Lawrence, D. M. (2026). Contrasting Parametric Sensitivities in Two Global Vegetation Models Using Parameter Perturbation Ensembles. Journal of Advances in Modeling Earth Systems, 18(3), e2025MS005590. https://doi.org/10.1029/2025MS005590

---

## Getting started with `meristem`

**meristem** is a Python toolkit for building and running parameter ensembles with the Community Land Model (CLM) and CLM-FATES. It turns a spreadsheet of parameter ranges into ready-to-run parameter files, using Latin Hypercube, one-at-a-time, or Morris sampling. It can then use those parameter files and submit and track an ensemble of CLM and CLM-FATES cases.
It can also chain the stages of a model run, such as spinup and transient, into PBS jobs that submit themselves one after another.

> A *meristem* is the growing tissue in plants that gives rise to new organs.

In this notebook you will:

1. Learn what `meristem` does and how it is organized
2. Install `meristem`
3. Confirm the installation works

### What `meristem` does

meristem has three parts:

| Module | Purpose | Where it runs |
|---|---|---|
| `meristem.paramgen` | Generate parameter files. Describe parameters and their ranges in a CSV, choose a sampling design, and get one parameter file per ensemble member plus a key of the sampled values. | Anywhere |
| `meristem.run` | Run one model run. Describe its stages (e.g. spinup → transient) in a YAML file and submit them as a chain of PBS jobs that advance on success and stop cleanly on failure. | PBS cluster with CESM/CTSM |
| `meristem.ensemble`| Run an ensemble: one run per paramgen parameter file, each with one or more stages, submitted and tracked together. | PBS cluster with CESM/CTSM |

#### Scope

`meristem` is developed for CLM and CLM-FATES, but its core is model-agnostic:

- **Sampling** works on normalized parameter values in [0, 1] and knows nothing about the model.
- **Parameter files** are read and written through a small `ParameterDataset` interface. NetCDF (CLM) and FATES JSON are currently supported.
- **Job chaining** runs each stage's setup script and submits the next stage on success. Job chaining currently requires a PBS scheduler.

### Installing `meristem`

meristem requires **Python 3.12 or newer**. It is not yet on PyPI or conda-forge, so you install it from source. It is currently housed in a personal GitHub repository, but the plan is to move it to an official NCAR repository once it is finalized.

#### Option 1: the provided conda environment (recommended)

Run these commands in a terminal (not in this notebook):

```bash
cd /glade/u/home/$USER/code
git clone https://github.com/adrifoster/meristem.git
cd meristem
conda env create -f environment.yml
conda activate meristem
```

This installs `meristem` in editable mode, along with its dependencies and the development tools (pytest, pylint, black).


#### Option 2: install into an existing environment

From the repository root, with your environment activated:

```bash
pip install -e .
```

### Running this notebook in the meristem environment

From here on, you should use JupyterLab to run the actual cells of this notebook. Follow the steps laid out in [our diagnostics tutorial](../diagnostics/diagnostics.ipynb) to launch Jupyter from the OnDemand website.

Then navigate to `notebooks/sensitivity` and open 

Once you are there, select `meristem` as the kernel.

### Check the installation

If the install worked, the cell below runs without errors.

In [ ]:
import meristem
from meristem.paramgen import ParamEnsemble

print("meristem imported from:", meristem.__file__)

### Next steps

Next we will explore how to use `meristem` to create suites of parameter files.